In [18]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import random

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [19]:

# ============================================================
# 1. Load CSV (unchanged)
# ============================================================

SEED = 42
EPOCHS = 200

np.random.seed(SEED)
torch.manual_seed(SEED)


In [20]:

df = pd.read_csv("cyber_dataset_C.csv")


In [21]:

df


,prompt,tool_ground_truth,tool_predicted,pred_svm,source,row_id
0,The vendor gave us a jump-box address and temp...,SSHConnect,SSHConnect,SSHConnect,pilot,C0000
1,Can you establish a terminal session to the Ub...,SSHConnect,SSHConnect,SSHConnect,pilot,C0001
2,We moved the inventory appliance to a new subn...,NmapScan,NmapScan,NmapScan,pilot,C0002
3,Does 172.16.9.25 still advertise that old SSH ...,NmapScan,NmapScan,NmapScan,pilot,C0003
4,The change request says the database node shou...,PortScan,PortScan,PortScan,pilot,C0004
...,...,...,...,...,...,...
1495,Read the local journal around the fan-control ...,ReadSyslog,ReadAuthLog,ReadSyslog,batch495,C1510
1496,The VPN interface renamed itself on this lapto...,ReadSyslog,GetSystemInfo,ReadSyslog,batch495,C1511
1497,After we rolled out a new eBPF-based observabi...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1512
1498,Get the operating-system journal entries from ...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1513


In [22]:

encoder = LabelEncoder()

y = encoder.fit_transform(df["tool_ground_truth"])


In [23]:

num_classes = len(encoder.classes_)

num_classes 


11

In [24]:

class_to_id = {
    name: i for i, name in enumerate(encoder.classes_)
}

class_to_id


{'BlockIP': 0,
 'CheckFailedLogins': 1,
 'CheckVulnerability': 2,
 'GetSystemInfo': 3,
 'ListProcesses': 4,
 'ListeningPorts': 5,
 'NmapScan': 6,
 'PortScan': 7,
 'ReadAuthLog': 8,
 'ReadSyslog': 9,
 'SSHConnect': 10}

In [25]:

y_svm = np.array([
    class_to_id[name]
    for name in df["pred_svm"]
])



y_svm.shape





(1500,)

In [26]:

y_svm


array([10, 10,  6, ...,  9,  9,  9])

In [27]:

y_llama = np.array([
    class_to_id.get(name, -1)
    for name in df["tool_predicted"]
])


In [28]:

# Unknown is encoded as -1.

set ( [  name for name in df["tool_predicted"]   ]  )


{'BlockIP',
 'CheckFailedLogins',
 'CheckVulnerability',
 'GetSystemInfo',
 'ListProcesses',
 'ListeningPorts',
 'NmapScan',
 'PortScan',
 'ReadAuthLog',
 'ReadSyslog',
 'SSHConnect',
 'Unknown'}

In [29]:

print(np.sum(y_llama == -1))


14


In [30]:

# ============================================================
# 2. Train / validation / test split
# ============================================================

indices = np.arange(len(df))

indices


array([   0,    1,    2, ..., 1497, 1498, 1499])

In [31]:

random_seed = int( random.random() * 100 )     ## 42

## X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)

trainval_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=random_seed,             ## SEED,
    stratify=y
)


In [32]:

print(  len(trainval_idx)  )
print(   len(test_idx   )  ) 


1200
300


In [33]:

test_idx


array([ 756,  557,  593, 1495, 1371, 1011,   61,  533,  149, 1285, 1416,
        455, 1319,  471,  888,  648,  192, 1122,  191,  148, 1379, 1499,
        514, 1100,   44, 1450,  159,  150, 1013, 1314, 1397,  485,  989,
         72,  654,  717, 1080,  176,  630,  811,  735, 1276, 1194,  949,
        526, 1316,  586,  259,  143,  298,  859,  262,  723,  210,  669,
       1094, 1077,  505,  394,  282,   65, 1479,  698,  345,  679,   52,
       1417,   97,  615,  851,  430, 1110,  846,  901,  252,  318,  852,
       1312,  207, 1145,  487,  413,  129, 1384, 1347,  897,  619, 1251,
        399, 1081,  450,  197, 1282, 1155,  100, 1364, 1051,  670, 1355,
       1266,  773,  151,  273,  253, 1418,  674,  801,  795,  577,  432,
       1129,  652,  870,  613,  990,  808,  153,  354,  900,  331, 1438,
        922, 1222, 1343,  545,  709, 1455,  959,   99,  217,  838,   20,
        112, 1182,  194, 1408,  415, 1205, 1106,  102,  974, 1000, 1163,
        883,  587, 1121, 1358,  130, 1464,  374,  2

In [34]:

train_idx = trainval_idx


In [35]:

print("Training:", len(train_idx))
print("Testing:", len(test_idx))



Training: 1200
Testing: 300


In [36]:

# ============================================================
# 3. Features: SVM + Llama predictions (23 features)
# ============================================================

# SVM has 11 classes
svm_onehot = np.zeros((len(df), num_classes))

svm_onehot.shape


(1500, 11)

In [37]:

y_svm


array([10, 10,  6, ...,  9,  9,  9])

In [38]:

valid_svm = y_svm >= 0

print(  np.sum( valid_svm )  )
print(     len( valid_svm )  )


1500
1500


In [39]:


svm_onehot[ 
    np.where(valid_svm)[0], y_svm[valid_svm] 
] = 1


In [40]:

for thing in svm_onehot:
    print(  thing  )


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 0

In [41]:

svm_onehot.shape


(1500, 11)

In [42]:

# Llama has 11 classes + Unknown
llama_onehot = np.zeros((len(df), num_classes + 1))

llama_onehot.shape


(1500, 12)

In [43]:

llama_idx = np.where(y_llama >= 0, y_llama, num_classes)


In [44]:


print(     len(  llama_idx )  )


1500


In [45]:

'''
import numpy as np

y_llama_rc = np.array([2, -1, 4, 0, -1])
num_classes_rc = 11

llama_idx_rc = np.where(y_llama_rc >= 0, y_llama_rc, num_classes_rc)

print("Original:", y_llama_rc)
print("Modified:", llama_idx_rc)

'''


'\nimport numpy as np\n\ny_llama_rc = np.array([2, -1, 4, 0, -1])\nnum_classes_rc = 11\n\nllama_idx_rc = np.where(y_llama_rc >= 0, y_llama_rc, num_classes_rc)\n\nprint("Original:", y_llama_rc)\nprint("Modified:", llama_idx_rc)\n\n'

In [46]:


llama_onehot[ 
      np.arange(len(df)), llama_idx
] = 1


In [47]:

for thing in llama_onehot:
    print( thing )


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1.

In [48]:


X = np.hstack((svm_onehot, llama_onehot)).astype(np.float32)

X.shape


(1500, 23)

In [49]:

X_train = torch.tensor( X[train_idx] )
X_test  = torch.tensor( X[test_idx]  )

print("Input features:", X.shape[1])


Input features: 23


In [50]:

print(  X_train.shape  )
print(  X_test.shape   )


torch.Size([1200, 23])
torch.Size([300, 23])


In [51]:

# ============================================================
# 4. Neural network targets (11 cybersecurity tools)
# ============================================================

target = y

target.shape


(1500,)

In [52]:


target_train = torch.tensor(
    target[train_idx],
    dtype=torch.long
)



In [53]:

target_train.shape


torch.Size([1200])

In [54]:

target_train


tensor([10,  5,  1,  ...,  4,  1,  8])

In [55]:

# ============================================================
# 5. Neural network (11 cybersecurity tools)
# ============================================================

class SelectorNet(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 11)
        )

    def forward(self, x):
        return self.net(x)


In [56]:



model = SelectorNet(  X.shape[1]  )




In [57]:

X.shape[1]


23

In [58]:

criterion = nn.CrossEntropyLoss()


In [59]:

optimizer = optim.Adam(model.parameters(), lr=0.001)


In [60]:



# ============================================================
# Train
# ============================================================

for epoch in range(EPOCHS):

    model.train()
    optimizer.zero_grad()

    logits = model(  X_train  )

    loss = criterion(logits, target_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss = {loss.item():.4f}"
        )





Epoch  20 | Loss = 2.3674
Epoch  40 | Loss = 2.3085
Epoch  60 | Loss = 2.2116
Epoch  80 | Loss = 2.0544
Epoch 100 | Loss = 1.8252
Epoch 120 | Loss = 1.5426
Epoch 140 | Loss = 1.2445
Epoch 160 | Loss = 0.9620
Epoch 180 | Loss = 0.7187
Epoch 200 | Loss = 0.5290


In [61]:

model.eval()

with torch.no_grad():
    ensemble_pred_res = torch.argmax(model(X_test), dim=1).numpy()


In [62]:

ensemble_pred_res.shape


(300,)

In [63]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true = y[test_idx]



In [64]:


print("Accuracy:", accuracy_score(y_true, ensemble_pred_res))

print("Macro F1:", f1_score(      y_true, ensemble_pred_res, average="macro" ))

print("Confusion Matrix:")
print(           confusion_matrix(y_true, ensemble_pred_res))

print("Classification Report:")
print(classification_report(      y_true, ensemble_pred_res,  zero_division=0   ))


Accuracy: 0.9333333333333333
Macro F1: 0.9330351106911735
Confusion Matrix:
[[28  0  0  0  0  0  0  0  0  0  0]
 [ 0 26  0  0  0  0  0  0  1  0  0]
 [ 0  0 26  0  0  0  0  0  0  0  1]
 [ 0  1  0 24  1  0  1  0  0  0  1]
 [ 0  1  0  0 26  0  0  0  0  0  0]
 [ 0  0  1  0  0 25  0  1  0  0  0]
 [ 0  1  0  0  0  0 25  1  0  0  1]
 [ 0  0  0  0  0  0  0 27  0  0  0]
 [ 0  0  0  0  2  1  0  0 23  0  1]
 [ 0  0  0  0  1  0  0  0  3 23  0]
 [ 0  0  0  0  0  0  0  0  0  0 27]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        28
           1       0.90      0.96      0.93        27
           2       0.96      0.96      0.96        27
           3       1.00      0.86      0.92        28
           4       0.87      0.96      0.91        27
           5       0.96      0.93      0.94        27
           6       0.96      0.89      0.93        28
           7       0.93      1.00      0.96        27
           8     

In [65]:

print("Accuracy:", accuracy_score(y_true, y_svm[test_idx]))

print("Macro F1:", f1_score(      y_true, y_svm[test_idx], average="macro" ))

print("Confusion Matrix:")
print(           confusion_matrix(y_true, y_svm[test_idx]))

print("Classification Report:")
print(classification_report(      y_true, y_svm[test_idx],  zero_division=0   ))


Accuracy: 0.93
Macro F1: 0.929828575892492
Confusion Matrix:
[[27  1  0  0  0  0  0  0  0  0  0]
 [ 0 24  0  1  0  0  0  0  2  0  0]
 [ 0  0 26  0  0  0  0  1  0  0  0]
 [ 0  1  0 25  1  0  0  0  0  1  0]
 [ 0  0  0  0 24  0  1  0  0  1  1]
 [ 0  0  1  0  0 25  0  1  0  0  0]
 [ 0  1  0  0  0  1 25  1  0  0  0]
 [ 0  0  0  0  0  0  0 27  0  0  0]
 [ 0  0  0  0  2  0  0  0 24  0  1]
 [ 0  0  0  0  1  0  0  0  1 25  0]
 [ 0  0  0  0  0  0  0  0  0  0 27]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98        28
           1       0.89      0.89      0.89        27
           2       0.96      0.96      0.96        27
           3       0.96      0.89      0.93        28
           4       0.86      0.89      0.87        27
           5       0.96      0.93      0.94        27
           6       0.96      0.89      0.93        28
           7       0.90      1.00      0.95        27
           8       0.89      0.8

In [66]:

print("Accuracy:", accuracy_score(y_true, y_llama[test_idx]  ))

print("Macro F1:", f1_score(      y_true, y_llama[test_idx], average="macro" ))

print("Confusion Matrix:")
print(           confusion_matrix(y_true, y_llama[test_idx]   ))

print("Classification Report:")
print(classification_report(      y_true, y_llama[test_idx]  ,  zero_division=0   ))


Accuracy: 0.8766666666666667
Macro F1: 0.8079621738310799
Confusion Matrix:
[[ 0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 28  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 26  0  0  0  0  0  0  1  0  0]
 [ 1  0  0 25  0  0  0  0  0  0  0  1]
 [ 0  0  0  0 27  0  0  1  0  0  0  0]
 [ 0  0  0  0  3 23  0  1  0  0  0  0]
 [ 0  0  0  0  0  0 13  5  9  0  0  0]
 [ 3  0  0  1  0  0  0 24  0  0  0  0]
 [ 0  0  0  0  0  0  0  5 22  0  0  0]
 [ 1  0  0  0  0  0  0  0  0 25  0  1]
 [ 0  0  0  0  1  0  0  0  0  3 23  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 27]]
Classification Report:
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00         0
           0       1.00      1.00      1.00        28
           1       1.00      0.96      0.98        27
           2       0.96      0.93      0.94        27
           3       0.87      0.96      0.92        28
           4       1.00      0.85      0.92        27
           5       1.00      0.48      0.65        27

In [67]:

print(f1_score(
    y[test_idx],
    y_llama[test_idx],
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
))


0.8814132805429963


In [68]:

print("Macro F1:", f1_score(
    y_true, y_llama[test_idx],
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    y_true, y_llama[test_idx],
    labels=np.arange(num_classes)
))

print("Classification Report:")
print(classification_report(
    y_true, y_llama[test_idx],
    labels=np.arange(num_classes),
    zero_division=0
))


Macro F1: 0.8814132805429963
Confusion Matrix:
[[28  0  0  0  0  0  0  0  0  0  0]
 [ 0 26  0  0  0  0  0  0  1  0  0]
 [ 0  0 25  0  0  0  0  0  0  0  1]
 [ 0  0  0 27  0  0  1  0  0  0  0]
 [ 0  0  0  3 23  0  1  0  0  0  0]
 [ 0  0  0  0  0 13  5  9  0  0  0]
 [ 0  0  1  0  0  0 24  0  0  0  0]
 [ 0  0  0  0  0  0  5 22  0  0  0]
 [ 0  0  0  0  0  0  0  0 25  0  1]
 [ 0  0  0  1  0  0  0  0  3 23  0]
 [ 0  0  0  0  0  0  0  0  0  0 27]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        28
           1       1.00      0.96      0.98        27
           2       0.96      0.93      0.94        27
           3       0.87      0.96      0.92        28
           4       1.00      0.85      0.92        27
           5       1.00      0.48      0.65        27
           6       0.67      0.86      0.75        28
           7       0.71      0.81      0.76        27
           8       0.86      0.93      0.89   


## NIO


In [ ]:

## !pip install matplotlib



## NIO - Case 1: Unknown


In [ ]:

# 14 cases where Llama predicted Unknown
unknown_idx = np.flatnonzero(y_llama == -1)

X_real = torch.tensor(X[unknown_idx], dtype=torch.float32)
y_true = torch.tensor(y[unknown_idx], dtype=torch.long)

# Freeze neural network
model.eval()
for p in model.parameters():
    p.requires_grad = False

# NIO: 14 NEW random input vectors (23 features each)
torch.manual_seed(42)
X_nio = torch.rand(14, 23, requires_grad=True)

optimizer = torch.optim.Adam([X_nio], lr=0.01)

for epoch in range(1000):
    optimizer.zero_grad()

    output = model(X_nio)
    loss = nn.CrossEntropyLoss()(output, y_true)

    loss.backward()
    optimizer.step()

    # Keep input features between 0 and 1
    with torch.no_grad():
        X_nio.clamp_(0, 1)

# Compare real and NIO-derived Unknown feature (feature 23)
real = X_real[:, 22].numpy()
nio = X_nio[:, 22].detach().numpy()

print("Ground truth:", y_true.numpy())
print("Real Unknown:", real)
print("NIO Unknown: ", nio)
print("Mean absolute error:", np.mean(np.abs(real - nio)))

# Compare all 23 features
print("\nMean squared error (all features):",
      torch.mean((X_real - X_nio.detach())**2).item())


In [ ]:

# Original SVM predictions (features 0–10)
svm_real = X_real[:, :11].argmax(dim=1).numpy()

# NIO-derived SVM predictions
svm_nio = X_nio[:, :11].detach().argmax(dim=1).numpy()

print("Real SVM: ", svm_real)
print("NIO SVM:  ", svm_nio)

# How many SVM features did NIO recover?
correct = np.sum(svm_real == svm_nio)

print("\nRecovered:", correct, "out of 14")
print("Accuracy:", correct / 14)

# Compare original and NIO-derived SVM feature vectors
print("SVM feature MSE:",
      torch.mean((X_real[:, :11] -
                  X_nio[:, :11].detach())**2).item())


In [ ]:

# Check whether NIO reproduces the 14 ground-truth outputs

with torch.no_grad():
    nio_predictions = model(X_nio).argmax(dim=1)

correct = (nio_predictions == y_true).sum().item()

print("Ground truth:   ", y_true.numpy())
print("NIO predictions:", nio_predictions.numpy())

print("\nCorrect:", correct, "out of 14")
print("Accuracy:", correct / 14)



## NIO - Agreement Cases 


In [ ]:

# Select 30 random agreement cases
np.random.seed(42)

agree_idx = np.flatnonzero(y_svm == y_llama)
idx = np.random.choice(agree_idx, 30, replace=False)

X_real = torch.tensor(X[idx], dtype=torch.float32)
y_true = torch.tensor(y[idx], dtype=torch.long)

# Freeze network
model.eval()
for p in model.parameters():
    p.requires_grad = False

# NIO from random inputs
torch.manual_seed(42)
X_nio = torch.rand(30, 23, requires_grad=True)

optimizer = torch.optim.Adam([X_nio], lr=0.01)

for epoch in range(1000):
    optimizer.zero_grad()
    loss = nn.CrossEntropyLoss()(model(X_nio), y_true)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        X_nio.clamp_(0, 1)

# Evaluate
with torch.no_grad():
    nio_pred = model(X_nio).argmax(dim=1)

svm_real = X_real[:, :11].argmax(dim=1)
svm_nio = X_nio[:, :11].detach().argmax(dim=1)

llama_real = X_real[:, 11:23].argmax(dim=1)
llama_nio = X_nio[:, 11:23].detach().argmax(dim=1)

print("Output recovery:", (nio_pred == y_true).sum().item(), "/30")
print("SVM recovery:", (svm_real == svm_nio).sum().item(), "/30")
print("Llama recovery:", (llama_real == llama_nio).sum().item(), "/30")
print("Input MSE:", torch.mean((X_real - X_nio.detach())**2).item())



## NIO - Disagreement Cases 


In [ ]:

# Select 30 random disagreement cases
np.random.seed(42)

disagree_idx = np.flatnonzero(y_svm != y_llama)
idx = np.random.choice(disagree_idx, 30, replace=False)

X_real = torch.tensor(X[idx], dtype=torch.float32)
y_true = torch.tensor(y[idx], dtype=torch.long)

# Freeze network
model.eval()
for p in model.parameters():
    p.requires_grad = False

# NIO from random inputs
torch.manual_seed(42)
X_nio = torch.rand(30, 23, requires_grad=True)

optimizer = torch.optim.Adam([X_nio], lr=0.01)

for epoch in range(1000):
    optimizer.zero_grad()
    loss = nn.CrossEntropyLoss()(model(X_nio), y_true)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        X_nio.clamp_(0, 1)

# Evaluate
with torch.no_grad():
    nio_pred = model(X_nio).argmax(dim=1)

svm_real = X_real[:, :11].argmax(dim=1)
svm_nio = X_nio[:, :11].detach().argmax(dim=1)

llama_real = X_real[:, 11:23].argmax(dim=1)
llama_nio = X_nio[:, 11:23].detach().argmax(dim=1)

print("Output recovery:", (nio_pred == y_true).sum().item(), "/30")
print("SVM recovery:", (svm_real == svm_nio).sum().item(), "/30")
print("Llama recovery:", (llama_real == llama_nio).sum().item(), "/30")
print("Input MSE:", torch.mean((X_real - X_nio.detach())**2).item())
